# Лабораторная работа: Collaborative Filtering

Цель: изучить методы collaborative filtering для рекомендаций, реализовать простую модель и оценить её качество.

## 1. Установка и импорт библиотек

In [108]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

## 2. Генерация синтетического датасета рейтингов пользователей

In [109]:
n_users = 50
n_items = 30
user_ids = [f'u{i}' for i in range(n_users)]
item_ids = [f'i{j}' for j in range(n_items)]
data = []
np.random.seed(42)
for u in user_ids:
    for i in np.random.choice(item_ids, size=10, replace=False):
        rating = np.random.randint(1,6)
        data.append([u, i, rating])
df = pd.DataFrame(data, columns=['user', 'item', 'rating'])
df.head()

,user,item,rating
0,u0,i27,2
1,u0,i15,4
2,u0,i23,4
3,u0,i17,3
4,u0,i8,4


In [110]:
df.rating.max()

5

## 3. Подготовка данных для Surprise

In [111]:
reader = Reader(rating_scale=(1,5))
dataset = Dataset.load_from_df(df[['user','item','rating']], reader)
trainset, testset = train_test_split(dataset, test_size=0.2, random_state=42)

## 4. Построение модели Collaborative Filtering (SVD)

In [112]:
algo = SVD(n_factors=10, n_epochs=20, lr_all=0.005, reg_all=0.02)
algo.fit(trainset)

## 5. Предсказания и оценка качества модели

In [113]:
predictions = algo.test(testset)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

RMSE: 1.4403
MAE:  1.2014


## 6. Пример предсказания рейтинга для конкретного пользователя и предмета

In [114]:
user_id = 'u0'
item_id = 'i0'
pred = algo.predict(user_id, item_id)
print(f'Predicted rating of user {user_id} for item {item_id}: {pred.est:.2f}')

Predicted rating of user u0 for item i0: 2.97


## 7. Получение топ-N рекомендаций для пользователя

In [115]:
def get_top_n_recommendations(algo, user_id, n=5):
    items_not_rated = [i for i in item_ids if not ((df['user']==user_id) & (df['item']==i)).any()]
    predictions = [algo.predict(user_id, i) for i in items_not_rated]
    predictions.sort(key=lambda x: x.est, reverse=True)
    top_n = predictions[:n]
    return [(p.iid, p.est) for p in top_n]
top5 = get_top_n_recommendations(algo, 'u0', n=5)
top5

[('i11', 3.9006603119771692),
 ('i16', 3.7598404681751445),
 ('i18', 3.6151643816738153),
 ('i20', 3.5261286404428165),
 ('i19', 3.368150882187657)]